<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/webscraping(w3d2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#given question:
#Extend into mini project: Web scraping agent that extracts news & emails daily updates

In [ ]:
#open ai api key
import os
openai_api_key = input("Please enter your OpenAI API key (e.g., sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx): ")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API key configured successfully.")


In [9]:
# 1. Imports and Setup
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime
import re

In [24]:
# 2. Configuration: Dynamic News Sources and Email Settings

NEWS_SOURCES = {}
while True:
    source_name = input("Enter news source name (e.g., TechCrunch, or 'done' to finish): ").strip()
    if source_name.lower() == 'done':
        break
    source_url = input(f"Enter URL for {source_name}: ").strip()
    print("Using default selectors for links, titles, and paragraphs. You can modify these in the code if needed.")
    NEWS_SOURCES[source_name] = {
        'url': source_url,
        'article_link_selector': 'a[href*="article"], a[href*="news"], a[class*="link"], a[class*="title"]', # Common link patterns
        'article_title_selector': 'h1, h2.headline, h1.article__title', # Common title patterns
        'article_content_selector': 'div.article-content p, div.article__body p, div.body p, p.paragraph' # Common content patterns
    }
    print(f"Added {source_name}: {source_url}\n")

if not NEWS_SOURCES:
    print("No news sources added. Exiting.")
else:
    print("--- News Sources Configured ---")
    for name, config in NEWS_SOURCES.items():
        print(f"- {name}: {config['url']}")
    print("-------------------------------")

# Email configuration
SENDER_EMAIL = input("\nEnter your sender email (e.g., your_email@gmail.com): ")
SENDER_PASSWORD = input("Enter your sender email app password (for Gmail, generate an 'App password' in Google Account Security settings): ")
RECIPIENT_EMAIL = input("Enter the recipient email: ")
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587

Enter news source name (e.g., TechCrunch, or 'done' to finish): toi
Enter URL for toi: https://timesofindia.indiatimes.com/sports/cricket/ipl/ipl-2026/ipl-2026-there-was-a-desire-sanju-samson-on-missing-ton-sunil-gavaskar-says-he-played-for-the-team/articleshow/130843485.cms
Using default selectors for links, titles, and paragraphs. You can modify these in the code if needed.
Added toi: https://timesofindia.indiatimes.com/sports/cricket/ipl/ipl-2026/ipl-2026-there-was-a-desire-sanju-samson-on-missing-ton-sunil-gavaskar-says-he-played-for-the-team/articleshow/130843485.cms

Enter news source name (e.g., TechCrunch, or 'done' to finish): done
--- News Sources Configured ---
- toi: https://timesofindia.indiatimes.com/sports/cricket/ipl/ipl-2026/ipl-2026-there-was-a-desire-sanju-samson-on-missing-ton-sunil-gavaskar-says-he-played-for-the-team/articleshow/130843485.cms
-------------------------------

Enter your sender email (e.g., your_email@gmail.com): jaddutemp@gmail.com
Enter your sende

In [25]:
# 3. Web Scraping Function
def scrape_news(source_name, source_config):
    """
    Scrapes recent articles from a given news source.
    Returns a list of dictionaries with article details.
    """
    articles = []
    print(f"\nScraping {source_name} from {source_config['url']}...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(source_config['url'], headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Limit to first few articles for conciseness
        for link_tag in soup.select(source_config['article_link_selector'])[:3]:
            article_url = link_tag.get('href')
            if not article_url or not article_url.startswith('http'):
                if article_url and not article_url.startswith('#'):
                    article_url = requests.compat.urljoin(source_config['url'], article_url)
                else:
                    continue

            try:
                article_response = requests.get(article_url, headers=headers, timeout=10)
                article_response.raise_for_status()
                article_soup = BeautifulSoup(article_response.text, 'html.parser')

                title_element = article_soup.select_one(source_config['article_title_selector'])
                title = title_element.get_text(strip=True) if title_element else 'No Title Found'

                content_elements = article_soup.select(source_config['article_content_selector'])
                content = ' '.join([p.get_text(strip=True) for p in content_elements if p.get_text(strip=True)])
                content = re.sub(r'\s+', ' ', content).strip()
                content = (content[:300] + '...') if len(content) > 300 else content

                if title != 'No Title Found' and len(content) > 50:
                    articles.append({
                        'source': source_name,
                        'title': title,
                        'url': article_url,
                        'content': content
                    })
            except requests.exceptions.RequestException as e:
                print(f"  Failed to fetch article {article_url}: {e}")
            except Exception as e:
                print(f"  Failed to parse article {article_url}: {e}")

    except requests.exceptions.RequestException as e:
        print(f"Failed to access {source_name} ({source_config['url']}): {e}")
    except Exception as e:
        print(f"An unexpected error occurred while scraping {source_name}: {e}")
    return articles

In [26]:
# 4. Email Sending Function
def send_email_update(recipient_email, articles):
    """
    Composes and sends an email with the scraped news articles.
    """
    if not articles:
        print("No articles found to send. Email not sent.")
        return

    msg = MIMEMultipart('alternative')
    msg['From'] = SENDER_EMAIL
    msg['To'] = recipient_email
    msg['Subject'] = f"Daily News Digest - {datetime.now().strftime('%Y-%m-%d')}"

    html_content = """
    <html><body>
        <h1>Your Daily News Digest</h1>
        <p>Here are the latest updates from your selected news sources:</p>
    """

    for article in articles:
        html_content += f"""
            <hr>
            <h2><a href="{article['url']}">{article['title']}</a></h2>
            <p><strong>Source:</strong> {article['source']}</p>
            <p>{article['content']}</p>
            <p><a href="{article['url']}">Read more</a></p>
        """
    html_content += """
        <hr>
        <p>This is an automated email. Do not reply.</p>
    </body></html>
    """

    msg.attach(MIMEText(html_content, 'html'))

    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls() # Secure the connection
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.send_message(msg)
        print(f"\nEmail update sent successfully to {recipient_email}!")
    except Exception as e:
        print(f"\nFailed to send email: {e}")
        print("Please check your sender email, app password, and recipient email details.")

In [ ]:
# 5. Main Execution

def run_news_agent():
    print("\n--- Starting Daily News Agent ---")
    all_news_articles = []

    for source_name, config in NEWS_SOURCES.items():
        scraped_articles = scrape_news(source_name, config)
        all_news_articles.extend(scraped_articles)

    print(f"Total unique articles scraped: {len(all_news_articles)}")

    send_email_update(RECIPIENT_EMAIL, all_news_articles)
    print("--- Daily News Agent Finished ---")

# Run the agent
if NEWS_SOURCES:
    run_news_agent()
else:
    print("Agent not run because no news sources were provided.")